In [8]:
import geopandas as gpd
import pandas as pd
from libpysal.weights import Queen
from esda.moran import Moran_Local
import numpy as np

In [9]:
# --- Parameters ---
shapefile_path = "../../data/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2)/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2).shp"  # Change to your shapefile path
id_column = "LAD25CD"       # Column where first letter is E/W/S/N
name_column = "LAD25NM"   # Column with local authority names

# --- Load shapefile ---
gdf = gpd.read_file(shapefile_path)


gdf = gdf[gdf['LAD25CD'] != 'E06000046'] # Exclude Isle of Wight as no neighbours
gdf = gdf[gdf['LAD25CD'] != 'E06000053'] # Exclude Isles of Scilly as no neighbours

# Filter to only England
gdf = gdf[gdf[id_column].str[0].isin(["E"])]

region = pd.read_csv('../../data/Region lookup.csv')

gdf = gdf.merge(
    region[['LAD24CD', 'RGN24NM']].rename(columns={'LAD24CD': 'LAD25CD', 'RGN24NM': 'Region'}),
    on='LAD25CD', 
    how='left'
)


Centroids and rook contiguity neighbours

In [10]:
# Ensure consistent projection
gdf = gdf.to_crs(epsg=27700)  # British National Grid

gdf['area_km2'] = gdf.geometry.area / 1e6

# Compute centroids (in projected CRS)
gdf["centroid_x"] = gdf.geometry.centroid.x
gdf["centroid_y"] = gdf.geometry.centroid.y

# Build neighbor list (rook contiguity, optimized with spatial index)
rows = []
for idx, area in gdf.iterrows():
    # Use spatial index for speed
    possible_matches_index = list(gdf.sindex.intersection(area.geometry.bounds))
    possible_matches = gdf.iloc[possible_matches_index]

    # Find actual touching neighbors
    touching = possible_matches[possible_matches.geometry.touches(area.geometry)]

    # Append one row per neighbor
    for _, neighbor in touching.iterrows():
        rows.append({
            id_column: area[id_column],
            name_column: area[name_column],
            "neighbour_name": neighbor[name_column],
            "neighbour_id": neighbor[id_column],
            "centroid_x": area["centroid_x"],
            "centroid_y": area["centroid_y"]
        })

# Create DataFrame in long format
neighbors_df = pd.DataFrame(rows)



Load in Average prices, reduce to just LAs in England and remove Islands

In [11]:
la_list = gdf[['LAD25CD']]

hpi_raw = pd.read_csv(filepath_or_buffer="../../data/UK-HPI-full-file-2025-05.csv")
hpi_raw = hpi_raw[['Date', 'RegionName', 'AreaCode', 'AveragePrice']]
hpi_raw = hpi_raw[hpi_raw['AreaCode'].isin(la_list['LAD25CD'])]
hpi_raw['Date'] = pd.to_datetime(hpi_raw['Date'], format='%d/%m/%Y')

hpi_raw = hpi_raw[hpi_raw['AreaCode'] != 'E06000046'] # Exclude Isle of Wight as no neighbours
hpi_raw = hpi_raw[hpi_raw['AreaCode'] != 'E06000053'] # Exclude Isles of Scilly as no neighbours

Calculate the average neighbours average value

In [12]:
la_neighbour_avg = []

for area in hpi_raw['AreaCode'].unique():
    area_neighbour_df = neighbors_df[neighbors_df['LAD25CD'] == area].copy()
    area_df = hpi_raw[hpi_raw['AreaCode'].isin(area_neighbour_df['neighbour_id'])].copy()

    neighbour_avg = area_df.groupby('Date').agg(
        AverageNeighbourPrice = ('AveragePrice', 'mean')
    ).reset_index()

    neighbour_avg['RegionName'] = area_neighbour_df['LAD25NM'].iloc[0]
    neighbour_avg['AreaCode'] = area_neighbour_df['LAD25CD'].iloc[0]

    # Store for concatenation
    la_neighbour_avg.append(neighbour_avg)

# Combine all area_code DataFrames into one
la_neighbour_avg = pd.concat(la_neighbour_avg, ignore_index=True)



In [13]:
hpi = pd.merge(
    hpi_raw,
    la_neighbour_avg,
    on=['Date', 'RegionName', 'AreaCode'],
    how='left'
)

In [14]:
w_queen = Queen.from_dataframe(gdf)  
w_queen.transform = 'r'  # Row-standardize weights

C:\Users\slong\AppData\Local\Temp\ipykernel_21736\3669738718.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w_queen = Queen.from_dataframe(gdf)


In [15]:
print(w_queen.neighbors)  # dict: {LA_index: [neighbor_indices]}
print(w_queen.weights)    # dict: {LA_index: [weights]}

{0: [3, 43], 1: [59, 2, 3], 2: [1, 59], 3: [0, 1, 4, 59, 43], 4: [59, 3, 43], 5: [45, 6, 235, 236, 237], 6: [5, 230, 233, 234, 44, 45, 237], 7: [225, 226, 135, 137, 141, 142, 143], 8: [136, 145], 9: [10], 10: [240, 9, 59, 12, 13], 11: [154, 12, 159], 12: [240, 168, 10, 11, 159], 13: [10, 59], 14: [73, 66, 70], 15: [152, 146, 147, 148], 16: [158, 148, 150, 55], 17: [169, 170, 173, 167], 18: [46, 101, 214], 19: [184, 46, 183], 20: [184, 185, 182], 21: [48, 22, 23, 24, 60], 22: [24, 21, 23], 23: [60, 21, 22], 24: [48, 100, 21, 22, 103], 25: [78], 26: [78, 79], 27: [48, 177, 100], 28: [64, 55, 157, 158, 63], 29: [50, 119], 30: [96, 90], 31: [89, 276, 87], 32: [130, 132, 126, 127], 33: [197, 110, 37, 38], 34: [176, 177, 114, 35, 48, 38, 105], 35: [176, 34, 38], 36: [37, 196, 277, 54], 37: [33, 195, 196, 197, 38, 36, 54], 38: [33, 34, 35, 37, 105, 110, 176, 54], 39: [49, 50, 54, 55, 56], 40: [210, 211, 84, 206], 41: [115, 108, 111], 42: [114, 107], 43: [0, 258, 3, 4, 51, 245, 58, 59], 44: [2

In [16]:
results = []

for month in hpi_raw['Date'].unique():
    month_data = hpi_raw[hpi_raw['Date'] == month]
    y = month_data['AveragePrice'].values
    
    moran_loc = Moran_Local(y, w_queen)
    
    month_results = pd.DataFrame({
        'local_I': moran_loc.Is,
        'p_value': moran_loc.p_sim,
        'quadrant': moran_loc.q
    }, index=month_data['AreaCode'].values)
    
    month_results['Date'] = month
    results.append(month_results)

local_moran_df = pd.concat(results)
local_moran_df.reset_index(inplace=True)

In [17]:
local_moran_df

hpi = pd.merge(
    hpi,
    local_moran_df,
    left_on=['Date', 'AreaCode'],
    right_on=['Date', 'index'],
    how='left'
).drop(columns=['index'])

In [18]:
hpi

,Date,RegionName,AreaCode,AveragePrice,AverageNeighbourPrice,local_I,p_value,quadrant
0,1995-01-01,Adur,E07000223,54669,58617.75,0.081166,0.252,3
1,1995-02-01,Adur,E07000223,55864,58719.75,0.049412,0.248,3
2,1995-03-01,Adur,E07000223,55880,59421.50,0.048777,0.263,3
3,1995-04-01,Adur,E07000223,55596,60068.00,0.059690,0.255,3
4,1995-05-01,Adur,E07000223,53483,60124.00,0.119831,0.221,3
...,...,...,...,...,...,...,...,...
107305,2025-01-01,York,E06000014,305903,243202.00,-0.084556,0.153,2
107306,2025-02-01,York,E06000014,304052,244940.50,-0.103360,0.142,2
107307,2025-03-01,York,E06000014,305832,248447.50,-0.103930,0.141,2
107308,2025-04-01,York,E06000014,307638,246783.50,-0.098423,0.133,2


In [19]:
CoL_centroid = gdf.query('LAD25CD == "E09000001"')[['centroid_x', 'centroid_y']]
gdf_attach = gdf[['LAD25CD', 'LAD25NM', 'area_km2', 'centroid_x', 'centroid_y', 'Region']]
gdf_attach.loc[:, 'CoL_centroid_x'] = CoL_centroid['centroid_x'].iloc[0]
gdf_attach.loc[:, 'CoL_centroid_y'] = CoL_centroid['centroid_y'].iloc[0]
gdf_attach['CoL_distance_km'] = np.hypot(
    gdf_attach['centroid_x'] - gdf_attach['CoL_centroid_x'],
    gdf_attach['centroid_y'] - gdf_attach['CoL_centroid_y']
) / 1000  # Convert to km   

C:\Users\slong\AppData\Local\Temp\ipykernel_21736\1814108756.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gdf_attach.loc[:, 'CoL_centroid_x'] = CoL_centroid['centroid_x'].iloc[0]
C:\Users\slong\AppData\Local\Temp\ipykernel_21736\1814108756.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gdf_attach.loc[:, 'CoL_centroid_y'] = CoL_centroid['centroid_y'].iloc[0]
C:\Users\slong\AppData\Local\Temp\ipykernel_21736\1814108756.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice

In [20]:
hpi = pd.merge(
    hpi,
    gdf_attach[['LAD25CD', 'area_km2', 'centroid_x', 'centroid_y', 'CoL_distance_km', 'Region']],
    left_on='AreaCode',
    right_on='LAD25CD',
    how='left'
).drop(columns=['LAD25CD'])

In [21]:
hpi_07 = hpi.query('Date > "2007-02-01"')

In [22]:
def compute_time_safe_fixed_effect(df, la_col="AreaCode", target_col="AveragePrice", date_col="Date"):
    """
    Computes a time-safe fixed effect for each row using only past information.
    """
    df = df.sort_values(date_col).copy()
    
    # Create storage
    fe_values = []
    
    # Running global mean
    running_sum = 0
    running_count = 0
    
    # Running LA-level sums
    la_sums = {}
    la_counts = {}
    
    for idx, row in df.iterrows():
        la = row[la_col]
        
        # Global FE
        global_mean = running_sum / running_count if running_count > 0 else row[target_col]
        la_mean = (la_sums.get(la, 0) / la_counts.get(la, 0)) if la in la_sums else global_mean
        
        fe = la_mean - global_mean
        fe_values.append(fe)
        
        # Add THIS row's target value for future calculations
        price = row[target_col]
        running_sum += price
        running_count += 1
        
        la_sums[la] = la_sums.get(la, 0) + price
        la_counts[la] = la_counts.get(la, 0) + 1
    
    df["LA_FE"] = fe_values
    return df

In [23]:

hpi_encode = compute_time_safe_fixed_effect(hpi_07)

hpi_encode = pd.get_dummies(hpi_encode, columns=["Region"], prefix="Region", drop_first=True, dtype=int)

hpi_encode

,Date,RegionName,AreaCode,AveragePrice,AverageNeighbourPrice,local_I,p_value,quadrant,area_km2,centroid_x,...,CoL_distance_km,LA_FE,Region_East of England,Region_London,Region_North East,Region_North West,Region_South East,Region_South West,Region_West Midlands,Region_Yorkshire and The Humber
146,2007-03-01,Adur,E07000223,202182,222076.250000,-0.044236,0.177,4,42.068486,519831.456946,...,75.909867,0.000000,0,0,0,0,1,0,0,0
83366,2007-03-01,St Albans,E07000240,335322,248187.142857,-0.588687,0.203,4,161.206621,514462.064765,...,33.448078,0.000000,1,0,0,0,0,0,0,0
83001,2007-03-01,Spelthorne,E07000213,239493,305757.714286,0.041802,0.391,1,51.171307,506612.189179,...,27.966480,0.000000,0,0,0,0,1,0,0,0
12191,2007-03-01,Broxbourne,E07000095,225959,260769.500000,0.154540,0.177,1,51.456854,534962.283891,...,23.592339,0.000000,1,0,0,0,0,0,0,0
82636,2007-03-01,Southwark,E09000028,319437,300105.250000,0.152483,0.348,1,28.877697,533820.180874,...,4.797720,0.000000,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71904,2025-05-01,Rossendale,E07000125,188034,170252.166667,0.353589,0.151,3,138.040939,382446.808235,...,283.622551,-123382.429937,0,0,0,1,0,0,0,0
72269,2025-05-01,Rother,E07000064,349166,357660.600000,0.048968,0.083,1,511.742707,578315.822888,...,76.487714,1059.082775,0,0,0,0,1,0,0,0
72634,2025-05-01,Rotherham,E08000018,193243,197394.166667,0.363607,0.136,3,286.534368,447551.630817,...,225.748728,-122146.526520,0,0,0,0,0,0,0,1
66794,2025-05-01,Oldham,E08000004,204063,213603.666667,0.362586,0.164,3,142.344982,396741.569262,...,262.660235,-121934.571801,0,0,0,1,0,0,0,0


In [25]:
sdlt_conditions = [
    #SDLT rules 03/2007 to 08/2008
    (hpi_encode["Date"].between("2007-03-01", "2008-08-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 125000),
    (hpi_encode["Date"].between("2007-03-01", "2008-08-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(125000, 250000, inclusive = "right")),
    (hpi_encode["Date"].between("2007-03-01", "2008-08-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(250000, 500000, inclusive = "right")),
    (hpi_encode["Date"].between("2007-03-01", "2008-08-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 500000),
    
    #SDLT rules 08/2008 to 12/2009
    (hpi_encode["Date"].between("2008-09-01", "2009-12-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 175000),
    (hpi_encode["Date"].between("2008-09-01", "2009-12-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(175000, 250000, inclusive = "right")),
    (hpi_encode["Date"].between("2008-09-01", "2009-12-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(250000, 500000, inclusive = "right")),
    (hpi_encode["Date"].between("2008-09-01", "2009-12-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 500000),
    
    #SDLT rules 01/2010 to 03/2011
    (hpi_encode["Date"].between("2010-01-01", "2011-03-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 125000),
    (hpi_encode["Date"].between("2010-01-01", "2011-03-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(125000, 250000, inclusive = "right")),
    (hpi_encode["Date"].between("2010-01-01", "2011-03-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(250000, 500000, inclusive = "right")),
    (hpi_encode["Date"].between("2010-01-01", "2011-03-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 500000),
    
    #SDLT rules 04/2011 to 11/2014
    (hpi_encode["Date"].between("2011-04-01", "2014-11-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 125000),
    (hpi_encode["Date"].between("2011-04-01", "2014-11-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(125000, 250000, inclusive = "right")),
    (hpi_encode["Date"].between("2011-04-01", "2014-11-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(250000, 500000, inclusive = "right")),
    (hpi_encode["Date"].between("2011-04-01", "2014-11-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(500000, 1e6, inclusive = "right")),
    (hpi_encode["Date"].between("2011-04-01", "2012-03-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 1e6),
    (hpi_encode["Date"].between("2012-04-01", "2014-11-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(1e6, 2e6, inclusive = "right")),
    (hpi_encode["Date"].between("2012-04-01", "2014-11-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 2e6),
    
    #SDLT rules 12/2014 to 06/2020
    (hpi_encode["Date"].between("2014-12-01", "2020-06-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 125000),
    (hpi_encode["Date"].between("2014-12-01", "2020-06-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(125000, 250000, inclusive = "right")),
    (hpi_encode["Date"].between("2014-12-01", "2020-06-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(250000, 925000, inclusive = "right")),
    (hpi_encode["Date"].between("2014-12-01", "2020-06-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(925000, 1.5e6, inclusive = "right")),
    (hpi_encode["Date"].between("2014-12-01", "2020-06-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 1.5e6),
    
    #SDLT rules 07/2020 to 06/2021
    (hpi_encode["Date"].between("2020-07-01", "2021-06-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 5e5),
    (hpi_encode["Date"].between("2020-07-01", "2021-06-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(5e5, 9.25e5, inclusive = "right")),
    (hpi_encode["Date"].between("2020-07-01", "2021-06-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(9.25e5, 1.5e6, inclusive = "right")),
    (hpi_encode["Date"].between("2020-07-01", "2021-06-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 1.5e6),
    
    #SDLT rules 07/2021 to 09/2021
    (hpi_encode["Date"].between("2021-07-01", "2021-09-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 2.5e5),
    (hpi_encode["Date"].between("2021-07-01", "2021-09-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(2.5e5, 9.25e5, inclusive = "right")),
    (hpi_encode["Date"].between("2021-07-01", "2021-09-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(9.25e5, 1.5e6, inclusive = "right")),
    (hpi_encode["Date"].between("2021-07-01", "2021-09-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 1.5e6),
    
    #SDLT rules 10/2021 to 09/2022
    (hpi_encode["Date"].between("2021-10-01", "2022-09-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 1.25e5),
    (hpi_encode["Date"].between("2021-10-01", "2022-09-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(1.25e5, 2.5e5, inclusive = "right")),
    (hpi_encode["Date"].between("2021-10-01", "2022-09-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(2.5e5, 9.25e5, inclusive = "right")),
    (hpi_encode["Date"].between("2021-10-01", "2022-09-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(9.25e5, 1.5e6, inclusive = "right")),
    (hpi_encode["Date"].between("2021-10-01", "2022-09-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 1.5e6),
    
    #SDLT rules 10/2022 to 03/2025
    (hpi_encode["Date"].between("2022-10-01", "2025-03-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 2.5e5),
    (hpi_encode["Date"].between("2022-10-01", "2025-03-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(2.5e5, 9.25e5, inclusive = "right")),
    (hpi_encode["Date"].between("2022-10-01", "2025-03-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(9.25e5, 1.5e6, inclusive = "right")),
    (hpi_encode["Date"].between("2022-10-01", "2025-03-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 1.5e6)
]

sdlt_choices = [
    #SDLT rules 03/2007 to 08/2008
    0, 1, 3, 4,

    #SDLT rules 08/2008 to 12/2009
    0, 1, 3, 4,

    #SDLT rules 01/2010 to 03/2011
    0, 1, 3, 4,

    #SDLT rules 04/2011 to 11/2014
    0, 1, 3, 4, 5, 5, 7,
    
    #SDLT rules 12/2014 to 06/2020
    0, 2, 5, 10, 12,

    #SDLT rules 07/2020 to 06/2021
    0, 5, 10, 12,

    #SDLT rules 07/2021 to 09/2021
    0, 5, 10, 12,

    #SDLT rules 10/2021 to 09/2022
    0, 2, 5, 10, 12,

    #SDLT rules 10/2022 to 03/2025
    0, 5, 10, 12
]

hpi_encode['sdlt_perc_threshold'] = np.select(sdlt_conditions, sdlt_choices)

In [26]:
hpi_encode

,Date,RegionName,AreaCode,AveragePrice,AverageNeighbourPrice,local_I,p_value,quadrant,area_km2,centroid_x,...,LA_FE,Region_East of England,Region_London,Region_North East,Region_North West,Region_South East,Region_South West,Region_West Midlands,Region_Yorkshire and The Humber,sdlt_perc_threshold
146,2007-03-01,Adur,E07000223,202182,222076.250000,-0.044236,0.177,4,42.068486,519831.456946,...,0.000000,0,0,0,0,1,0,0,0,1
83366,2007-03-01,St Albans,E07000240,335322,248187.142857,-0.588687,0.203,4,161.206621,514462.064765,...,0.000000,1,0,0,0,0,0,0,0,3
83001,2007-03-01,Spelthorne,E07000213,239493,305757.714286,0.041802,0.391,1,51.171307,506612.189179,...,0.000000,0,0,0,0,1,0,0,0,1
12191,2007-03-01,Broxbourne,E07000095,225959,260769.500000,0.154540,0.177,1,51.456854,534962.283891,...,0.000000,1,0,0,0,0,0,0,0,1
82636,2007-03-01,Southwark,E09000028,319437,300105.250000,0.152483,0.348,1,28.877697,533820.180874,...,0.000000,0,1,0,0,0,0,0,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71904,2025-05-01,Rossendale,E07000125,188034,170252.166667,0.353589,0.151,3,138.040939,382446.808235,...,-123382.429937,0,0,0,1,0,0,0,0,0
72269,2025-05-01,Rother,E07000064,349166,357660.600000,0.048968,0.083,1,511.742707,578315.822888,...,1059.082775,0,0,0,0,1,0,0,0,0
72634,2025-05-01,Rotherham,E08000018,193243,197394.166667,0.363607,0.136,3,286.534368,447551.630817,...,-122146.526520,0,0,0,0,0,0,0,1,0
66794,2025-05-01,Oldham,E08000004,204063,213603.666667,0.362586,0.164,3,142.344982,396741.569262,...,-121934.571801,0,0,0,1,0,0,0,0,0
